# Claude API를 활용한 Programatic Tool Calling (PTC)

Programmatic Tool Calling (PTC)은 각 tool 호출마다 모델을 거치는 round-trip을 요구하는 대신, Claude가 Code Execution 환경 안에서 programmatically tool을 호출하는 코드를 작성할 수 있게 합니다. 이는 여러 tool 호출의 end-to-end latency를 크게 줄이며, 모델의 context window에 들어오기 전에 관련 없는 context를 제거하는 코드를 모델이 작성할 수 있게 하여 token 소모를 획기적으로 줄일 수 있습니다(예: 크고 잡음이 많은 파일 안에서 grep으로 핵심 정보 찾기).

직접 수정할 수 없는 third-party API와 tool을 다룰 때, PTC는 Claude가 Code Execution 환경에서 호출 가능한 코드를 작성하도록 하여 context 사용량을 줄이는 데 도움이 됩니다.

이 cookbook에서는 팀 경비 관리를 위한 mock API를 다룹니다. 이 API는 여러 번의 호출이 필요하도록 설계되었으며, Programmatic Tool Calling의 이점을 보여주는 큰 결과를 반환합니다.


## 이 cookbook을 마치면 다음을 할 수 있습니다.

- 일반적인 tool calling과 programatic tool calling (PTC)의 차이 이해하기
- PTC를 활용하는 agent 작성하기



## 사전 요구사항

이 guide를 따라 하기 전에 다음을 준비하세요.

**필요한 지식**

- Python 기초 - async/await, 함수, 기본 자료구조를 익숙하게 사용할 수 있어야 합니다.
- agentic 패턴과 tool calling에 대한 기본 이해

**필요한 도구**

- Python 3.11 이상
- Anthropic API key
- Anthropic Python SDK >= 0.72



## 설정

먼저 필요한 dependencies를 설치합니다.


In [ ]:
# %pip install -r requirements.txt

참고: .env 파일에 다음이 포함되어 있는지 확인합니다.

`ANTHROPIC_API_KEY=your_key_here`

환경 변수를 불러오고 client를 설정합니다. Claude message 응답을 시각화하는 helper utility도 불러옵니다.



In [2]:
from dotenv import load_dotenv
from utils.visualize import visualize

load_dotenv()

MODEL = "claude-sonnet-4-6"

viz = visualize(auto_show=True)

## Third-Party API 이해하기

[utils/team_expense_api.py](utils/team_expense_api.py)에는 `get_team_members`, `get_expenses`, `get_custom_budget` 세 함수가 정의되어 있습니다. `get_team_members` 함수는 지정한 부서의 모든 직원과 그들의 역할, level, 연락처 정보를 가져올 수 있게 합니다. `get_expenses` 함수는 특정 분기에 대한 직원의 모든 경비 항목을 반환합니다. 직원당 수백 개의 record가 있을 수 있으며, 각 record에는 영수증 URL, 승인 chain, 가맹점 세부사항 등을 포함하는 광범위한 metadata가 담겨 있습니다. `get_custom_budget` 함수는 특정 직원에게 사용자 정의 출장 예산 예외가 있는지 확인합니다(그렇지 않으면 표준 분기 한도 $5,000를 사용합니다).

이 시나리오에서는 팀 경비를 분석하여 어떤 직원이 예산을 초과했는지 파악해야 합니다. 기존 방식으로는 각 사람의 경비 보고서를 수동으로 가져와 범주별 경비를 합산하고, 예산 한도와 비교하고(사용자 정의 예산 예외 확인), 보고서를 작성할 수 있습니다. 대신 사용 가능한 tool로 팀 데이터를 가져오고 풍부한 metadata를 포함한 수백 개의 경비 항목을 조회하여 누가 예산을 초과했는지 판단하도록 Claude에 분석을 요청하겠습니다.

핵심 어려움은 각 직원에게 가져오고 parsing하고 집계해야 할 100개 이상의 경비 항목이 있을 수 있으며, `get_custom_budget` tool은 경비를 분석하여 누군가 표준 예산을 초과했는지 확인한 후에만 호출할 수 있다는 점입니다. 이는 순차적인 dependency chain을 형성하므로 Programmatic Tool Calling의 이점을 시연하기에 이상적인 사용 사례입니다.

Tool 정의를 messages API에 전달하고 Claude에 분석을 요청하겠습니다. Claude API의 tool use 동작을 잘 모른다면 [tool use 구현](https://docs.claude.com/en/docs/agents-and-tools/tool-use/implement-tool-use) 문서를 읽어 보세요.


In [3]:
import json

import anthropic
from utils.team_expense_api import get_custom_budget, get_expenses, get_team_members

client = anthropic.Anthropic()

# 팀 경비 API의 tool 정의
tools = [
    {
        "name": "get_team_members",
        "description": 'Returns a list of team members for a given department. Each team member includes their ID, name, role, level (junior, mid, senior, staff, principal), and contact information. Use this to get a list of people whose expenses you want to analyze. Available departments are: engineering, sales, and marketing.\n\nRETURN FORMAT: Returns a JSON string containing an ARRAY of team member objects (not wrapped in an outer object). Parse with json.loads() to get a list. Example: [{"id": "ENG001", "name": "Alice", ...}, {"id": "ENG002", ...}]',
        "input_schema": {
            "type": "object",
            "properties": {
                "department": {
                    "type": "string",
                    "description": "The department name. Case-insensitive.",
                }
            },
            "required": ["department"],
        },
        "input_examples": [
            {"department": "engineering"},
            {"department": "sales"},
            {"department": "marketing"},
        ],
    },
    {
        "name": "get_expenses",
        "description": "Returns all expense line items for a given employee in a specific quarter. Each expense includes extensive metadata: date, category, description, amount (in USD), currency, status (approved, pending, rejected), receipt URL, approval chain, merchant name and location, payment method, and project codes. An employee may have 20-50+ expense line items per quarter, and each line item contains substantial metadata for audit and compliance purposes. Categories include: 'travel' (flights, trains, rental cars, taxis, parking), 'lodging' (hotels, airbnb), 'meals', 'software', 'equipment', 'conference', 'office', and 'internet'. IMPORTANT: Only expenses with status='approved' should be counted toward budget limits.\n\nRETURN FORMAT: Returns a JSON string containing an ARRAY of expense objects (not wrapped in an outer object with an 'expenses' key). Parse with json.loads() to get a list directly. Example: [{\"expense_id\": \"ENG001_Q3_001\", \"amount\": 1250.50, \"category\": \"travel\", ...}, {...}]",
        "input_schema": {
            "type": "object",
            "properties": {
                "employee_id": {
                    "type": "string",
                    "description": "The unique employee identifier",
                },
                "quarter": {
                    "type": "string",
                    "description": "Quarter identifier: 'Q1', 'Q2', 'Q3', or 'Q4'",
                },
            },
            "required": ["employee_id", "quarter"],
        },
        "input_examples": [
            {"employee_id": "ENG001", "quarter": "Q3"},
            {"employee_id": "SAL002", "quarter": "Q1"},
            {"employee_id": "MKT001", "quarter": "Q4"},
        ],
    },
    {
        "name": "get_custom_budget",
        "description": 'Get the custom quarterly travel budget for a specific employee. Most employees have a standard $5,000 quarterly travel budget. However, some employees have custom budget exceptions based on their role requirements. This function checks if a specific employee has a custom budget assigned.\n\nRETURN FORMAT: Returns a JSON string containing a SINGLE OBJECT (not an array). Parse with json.loads() to get a dict. Example: {"user_id": "ENG001", "has_custom_budget": false, "travel_budget": 5000, "reason": "Standard", "currency": "USD"}',
        "input_schema": {
            "type": "object",
            "properties": {
                "user_id": {
                    "type": "string",
                    "description": "The unique employee identifier",
                }
            },
            "required": ["user_id"],
        },
        "input_examples": [
            {"user_id": "ENG001"},
            {"user_id": "SAL002"},
            {"user_id": "MKT001"},
        ],
    },
]

tool_functions = {
    "get_team_members": get_team_members,
    "get_expenses": get_expenses,
    "get_custom_budget": get_custom_budget,
}

## 기존 Tool Calling (Baseline)

첫 번째 예제에서는 기존 tool calling을 사용하여 baseline을 설정하겠습니다.

초기 query로 `messages.create` API를 호출합니다. 모델이 `tool_use` 사유로 멈추면 요청대로 tool을 실행한 다음, tool output을 messages에 추가하고 모델을 다시 호출합니다.


In [4]:
import time

from anthropic.types import TextBlock, ToolUseBlock
from anthropic.types.beta import (
    BetaMessageParam as MessageParam,
)
from anthropic.types.beta import (
    BetaTextBlock,
    BetaToolUseBlock,
)

messages: list[MessageParam] = []


def run_agent_without_ptc(user_message):
    """기존 tool calling으로 agent 실행"""
    messages.append({"role": "user", "content": user_message})
    total_tokens = 0
    start_time = time.time()
    api_counter = 0

    while True:
        response = client.beta.messages.create(
            model=MODEL,
            max_tokens=4000,
            tools=tools,
            messages=messages,
            betas=["advanced-tool-use-2025-11-20"],
        )

        api_counter += 1

        # Token 사용량 추적
        total_tokens += response.usage.input_tokens + response.usage.output_tokens
        viz.capture(response)
        if response.stop_reason == "end_turn":
            # 응답에서 첫 text block 추출
            final_response = next(
                (
                    block.text
                    for block in response.content
                    if isinstance(block, (BetaTextBlock, TextBlock))
                ),
                None,
            )
            elapsed_time = time.time() - start_time
            return final_response, messages, total_tokens, elapsed_time, api_counter

        # Tool 호출 처리
        if response.stop_reason == "tool_use":
            # 먼저 assistant 응답을 messages에 추가
            messages.append({"role": "assistant", "content": response.content})

            # 모든 tool 결과 수집
            tool_results = []

            for block in response.content:
                if isinstance(block, (BetaToolUseBlock, ToolUseBlock)):
                    tool_name = block.name
                    tool_input = block.input
                    tool_use_id = block.id

                    result = tool_functions[tool_name](**tool_input)

                    content = str(result)

                    tool_result = {
                        "type": "tool_result",
                        "tool_use_id": tool_use_id,
                        "content": content,
                    }
                    tool_results.append(tool_result)

            # 수집 후 모든 tool 결과를 한 번에 추가
            messages.append({"role": "user", "content": tool_results})

        else:
            print(f"\nUnexpected stop reason: {response.stop_reason}")
            elapsed_time = time.time() - start_time

            final_response = next(
                (
                    block.text
                    for block in response.content
                    if isinstance(block, (BetaTextBlock, TextBlock))
                ),
                f"Stopped with reason: {response.stop_reason}",
            )
            return final_response, messages, total_tokens, elapsed_time, api_counter

모델에 보내는 초기 query에는 모델을 유도하는 지시가 있습니다. 간결함을 위해 각 tool을 한 번만 호출하도록 요청했습니다. 더 깊이 조사할 때는 모델이 여러 시스템이나 기간을 살펴보려 할 수 있습니다.


In [5]:
query = "Which engineering team members exceeded their Q3 travel budget? Standard quarterly travel budget is $5,000. However, some employees have custom budget limits. For anyone who exceeded the $5,000 standard budget, check if they have a custom budget exception. If they do, use that custom limit instead to determine if they truly exceeded their budget."

In [6]:
# Agent 실행
result, conversation, total_tokens, elapsed_time, api_count_without_ptc = run_agent_without_ptc(
    query
)

print(f"Result: {result}")
print(f"API calls made: {api_count_without_ptc}")
print(f"Total tokens used: {total_tokens:,}")
print(f"Total time taken: {elapsed_time:.2f}s")

╭────────────────────────────────────────────── Claude API Response ──────────────────────────────────────────────╮
│ Claude Message (assistant) │ tokens: 1,859 in • 85 out • 1,944 total                                            │
│ ├── Model: claude-sonnet-4-6                                                                           │
│ ├── Stop Reason: tool_use                                                                                       │
│ └── Content (2 blocks)                                                                                          │
│     ├── Block 1                                                                                                 │
│     │   └── Text                                                                                                │
│     │       └── I'll help you identify which engineering team members exceeded their Q3 travel budget. Let me   │
│     │           start by getting the list of engineering team members.                                          │
│     └── Block 2                                                                                                 │
│         └── Tool Use: get_team_members                                                                          │
│             ├── ID: toolu_01LuouuJYp1sSvBe2Du7EG7v                                                              │
│             ├── Caller: model (direct)                                                                          │
│             └── Input:                                                                                          │
│                 └── {                                                                                           │
│                       "department": "engineering"                                                               │
│                     }                                                                                           │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────── Claude API Response ───────────────────────────────╮
│ Claude Message (assistant) │ tokens: 2,473 in • 497 out • 2,970 total            │
│ ├── Model: claude-sonnet-4-6                                            │
│ ├── Stop Reason: tool_use                                                        │
│ └── Content (9 blocks)                                                           │
│     ├── Block 1                                                                  │
│     │   └── Text                                                                 │
│     │       └── Now let me get the Q3 expenses for all engineering team members: │
│     ├── Block 2                                                                  │
│     │   └── Tool Use: get_expenses                                               │
│     │       ├── ID: toolu_01Wu8LLTT2sKTTqpVwGT65Lj                               │
│     │       ├── Caller: model (direct)                                           │
│     │       └── Input:                                                           │
│     │           └── {                                                            │
│     │                 "employee_id": "ENG001",                                   │
│     │                 "quarter": "Q3"                                            │
│     │               }                                                            │
│     ├── Block 3                                                                  │
│     │   └── Tool Use: get_expenses                                               │
│     │       ├── ID: toolu_01KzjQ5mQJa9ocWjCGzYkD9F                               │
│     │       ├── Caller: model (direct)                                           │
│     │       └── Input:                                                           │
│     │           └── {                                                            │
│     │                 "employee_id": "ENG002",                                   │
│     │                 "quarter": "Q3"                                            │
│     │               }                                                            │
│     ├── Block 4                                                                  │
│     │   └── Tool Use: get_expenses                                               │
│     │       ├── ID: toolu_01RjjhZTg9JsKXE5E9S6Foho                               │
│     │       ├── Caller: model (direct)                                           │
│     │       └── Input:                                                           │
│     │           └── {                                                            │
│     │                 "employee_id": "ENG003",                                   │
│     │                 "quarter": "Q3"                                            │
│     │               }                                                            │
│     ├── Block 5                                                                  │
│     │   └── Tool Use: get_expenses                                               │
│     │       ├── ID: toolu_013xqpxpfc2N9rP5W5uMLAo9                               │
│     │       ├── Caller: model (direct)                                           │
│     │       └── Input:                                                           │
│     │           └── {                                                            │
│     │                 "employee_id": "ENG004",                                   │
│     │                 "quarter": "Q3"                                            │
│     │               }                                                            │
│     ├── Block 6                                                                  │
│     │   └── Tool Use: get_expenses                                               │
│     │       ├── ID: toolu_019zfzG6Wox8iDqy1dUXiH3t                               │
│     │       

╭────────────────────────────────────────────── Claude API Response ──────────────────────────────────────────────╮
│ Claude Message (assistant) │ tokens: 51,744 in • 290 out • 52,034 total                                         │
│ ├── Model: claude-sonnet-4-6                                                                           │
│ ├── Stop Reason: tool_use                                                                                       │
│ └── Content (7 blocks)                                                                                          │
│     ├── Block 1                                                                                                 │
│     │   └── Text                                                                                                │
│     │       └── Now let me calculate the approved travel expenses for each engineer and identify who exceeded   │
│     │           $5,000:                                                                                         │
│     ├── Block 2                                                                                                 │
│     │   └── Tool Use: get_custom_budget                                                                         │
│     │       ├── ID: toolu_013oegKwjvToLwEW1daDD8av                                                              │
│     │       ├── Caller: model (direct)                                                                          │
│     │       └── Input:                                                                                          │
│     │           └── {                                                                                           │
│     │                 "user_id": "ENG001"                                                                       │
│     │               }                                                                                           │
│     ├── Block 3                                                                                                 │
│     │   └── Tool Use: get_custom_budget                                                                         │
│     │       ├── ID: toolu_0162W4Ycr9FcVVED65exjAj4                                                              │
│     │       ├── Caller: model (direct)                                                                          │
│     │       └── Input:                                                                                          │
│     │           └── {                                                                                           │
│     │                 "user_id": "ENG003"                                                                       │
│     │               }                                                                                           │
│     ├── Block 4                                                                                                 │
│     │   └── Tool Use: get_custom_budget                                                                         │
│     │       ├── ID: toolu_01JcTX5rnwFxA99Am33gXmh6                                                              │
│     │       ├── Caller: model (direct)                                                                          │
│     │       └── Input:                                                                                          │
│     │           └── {                                                                                           │
│     │                 "user_id": "ENG005"                                                                       │
│     │               }                                                                                           │
│     ├── Block 5                                                                                                 │
│     │   └── Tool Use: get_custom_budget                        

╭────────────────────────────────────────────── Claude API Response ──────────────────────────────────────────────╮
│ Claude Message (assistant) │ tokens: 52,533 in • 992 out • 53,525 total                                         │
│ ├── Model: claude-sonnet-4-6                                                                           │
│ ├── Stop Reason: end_turn                                                                                       │
│ └── Content (1 blocks)                                                                                          │
│     └── Block 1                                                                                                 │
│         └── Text                                                                                                │
│             └── Now let me analyze the data. I'll calculate the approved travel expenses for each engineer:     │
│                                                                                                                 │
│                 **Analysis of Q3 Travel Expenses:**                                                             │
│                                                                                                                 │
│                 **ENG001 - Alice Chen (Senior Software Engineer)**                                              │
│                 - Approved travel expenses: $1,161.04 + $18.63 + $13.21 + $36.55 + $1,440.42 + $166.46 + $48.43 │
│                 + $1,124.56 + $1,245.90 + $1,498.42 = **$6,753.62**                                             │
│                 - Budget: $5,000 (Standard)                                                                     │
│                 - **EXCEEDED by $1,753.62** ❌                                                                  │
│                                                                                                                 │
│                 **ENG002 - Bob Martinez (Staff Engineer)**                                                      │
│                 - Approved travel expenses: $180.16 + $10.07 + $20.76 = **$210.99**                             │
│                 - Budget: $5,000 (Standard)                                                                     │
│                 - Under budget ✓                                                                                │
│                                                                                                                 │
│                 **ENG003 - Carol White (Software Engineer)**                                                    │
│                 - Approved travel expenses: $24.75 + $424.74 + $1,397.17 + $1,026.12 + $1,288.36 + $1,128.90 +  │
│                 $1,148.42 + $45.03 = **$6,483.49**                                                              │
│                 - Budget: $5,000 (Standard)                                                                     │
│                 - **EXCEEDED by $1,483.49** ❌                                                                  │
│                                                                                                                 │
│                 **ENG004 - David Kim (Principal Engineer)**                                                     │
│                 - Approved travel expenses: $21.68 + $46.12 + $1,008.68 + $46.43 = **$1,122.91**                │
│                 - Budget: $5,000 (Standard)                                                                     │
│                 - Under budget ✓                                                                                │
│                                                                                                                 │
│                 **ENG005 - Emma Johnson (Junior Software Engineer)                                              │
│                 ... (truncated)                                  

Result: Now let me analyze the data. I'll calculate the approved travel expenses for each engineer:

**Analysis of Q3 Travel Expenses:**

**ENG001 - Alice Chen (Senior Software Engineer)**
- Approved travel expenses: $1,161.04 + $18.63 + $13.21 + $36.55 + $1,440.42 + $166.46 + $48.43 + $1,124.56 + $1,245.90 + $1,498.42 = **$6,753.62**
- Budget: $5,000 (Standard)
- **EXCEEDED by $1,753.62** ❌

**ENG002 - Bob Martinez (Staff Engineer)**
- Approved travel expenses: $180.16 + $10.07 + $20.76 = **$210.99**
- Budget: $5,000 (Standard)
- Under budget ✓

**ENG003 - Carol White (Software Engineer)**
- Approved travel expenses: $24.75 + $424.74 + $1,397.17 + $1,026.12 + $1,288.36 + $1,128.90 + $1,148.42 + $45.03 = **$6,483.49**
- Budget: $5,000 (Standard)
- **EXCEEDED by $1,483.49** ❌

**ENG004 - David Kim (Principal Engineer)**
- Approved travel expenses: $21.68 + $46.12 + $1,008.68 + $46.43 = **$1,122.91**
- Budget: $5,000 (Standard)
- Under budget ✓

**ENG005 - Emma Johnson (Junior Software E

좋습니다! Claude가 사용 가능한 tool을 성공적으로 사용하여 출장 예산을 초과한 팀원을 파악한 것을 볼 수 있습니다. 하지만 이 작업에 많은 token을 사용한 것도 볼 수 있습니다. Claude는 모든 경비 항목을 parsing하고 범주별 합계를 구하여 예산 한도와 비교하기 위해 context window로 받아들여야 했습니다. 직원당 100개 이상의 record가 있을 수 있으며, 각각 영수증 URL, 승인 chain, 가맹점 정보 등을 포함한 광범위한 metadata가 있습니다.

또한 기존 tool calling 접근법은 여러 번의 순차적 round-trip을 필요로 합니다. 먼저 팀원을 가져오고, 각 사람의 경비를 가져온 다음, 표준 한도를 초과한 사람의 사용자 정의 예산을 확인합니다. 각 round-trip은 latency를 추가하며, 경비 record의 모든 풍부한 metadata가 모델의 context를 거칩니다.

대신 Claude가 code execution 환경에서 이러한 큰 dataset을 처리하는 코드를 작성하도록 하여 PTC로 성능을 개선할 수 있는지 살펴보겠습니다.


Tool에 PTC를 활성화하려면 먼저 code execution으로 호출 가능해야 하는 모든 tool에 `allowed_callers` field를 추가해야 합니다.

**고려할 핵심 사항**

- allowed_callers가 없는 tool은 기본적으로 모델만 호출할 수 있습니다.
- 여러 caller를 포함하면 모델과 code execution 모두 tool을 호출할 수 있습니다: `["direct", "code_execution_20250825"]`
- programmatic/반복 실행에 안전한 tool만 활성화하세요.



In [7]:
import copy

ptc_tools = copy.deepcopy(tools)
for tool in ptc_tools:
    tool["allowed_callers"] = ["code_execution_20250825"]  # type: ignore


# Code execution tool 추가
ptc_tools.append(
    {
        "type": "code_execution_20250825",  # type: ignore
        "name": "code_execution",
    }
)

Programmatic tool calling을 허용하도록 tool 정의를 업데이트했으므로 이제 PTC로 agent를 실행할 수 있습니다. 이를 위해 함수를 몇 군데 변경해야 했습니다. `beta` messages API를 사용해야 합니다.

1. betas에 `"advanced-tool-use-2025-11-20"`를 추가했습니다.
2. 요청에 `container_id`가 정의되어 있으면 이를 전달합니다. 이는 이 예제처럼 stateful workflow일 때만 필요합니다. Single-turn workflow에서는 필요하지 않습니다.
3. `tool_use` block의 `caller` field를 확인하여 이 tool 호출이 직접 모델 호출인지 programmatic 호출인지 판단할 수 있습니다.

두 경우 모두 Claude API로 tool 결과를 전송하지만, 모델은 `direct` 호출만 "본다"는 점에 유의하세요. `code_execution_20250825` 유형은 code execution container만 봅니다.


In [8]:
messages = []


def run_agent_with_ptc(user_message):
    """PTC로 agent 실행"""
    messages.append({"role": "user", "content": user_message})
    total_tokens = 0
    start_time = time.time()
    container_id = None
    api_counter = 0

    while True:
        # PTC beta header로 요청 구성
        request_params = {
            "model": MODEL,
            "max_tokens": 4000,
            "tools": ptc_tools,
            "messages": messages,
        }

        response = client.beta.messages.create(
            **request_params,
            betas=[
                "advanced-tool-use-2025-11-20",
            ],
            extra_body={"container": container_id} if container_id else None,
        )
        viz.capture(response)
        api_counter += 1

        # Stateful 실행을 위한 container 추적
        if hasattr(response, "container") and response.container:
            container_id = response.container.id
            print(f"\n[Container] ID: {container_id}")
            if hasattr(response.container, "expires_at"):
                # Container가 만료되면 workflow를 재시작해야 합니다. 이 예제에서는 만료 전에 완료됩니다.
                print(f"[Container] Expires at: {response.container.expires_at}")

        # Token 사용량 추적
        total_tokens += response.usage.input_tokens + response.usage.output_tokens

        if response.stop_reason == "end_turn":
            # 응답에서 첫 text block 추출
            final_response = next(
                (block.text for block in response.content if isinstance(block, BetaTextBlock)),
                None,
            )
            elapsed_time = time.time() - start_time
            return final_response, messages, total_tokens, elapsed_time, api_counter

        # 이전처럼 tool 호출 처리
        if response.stop_reason == "tool_use":
            # 먼저 assistant 응답을 messages에 추가
            messages.append({"role": "assistant", "content": response.content})

            # 모든 tool 결과 수집
            tool_results = []

            for block in response.content:
                if isinstance(block, BetaToolUseBlock):
                    tool_name = block.name
                    tool_input = block.input
                    tool_use_id = block.id

                    # Caller 유형으로 tool이 호출된 방식 파악 가능
                    caller_type = block.caller["type"]  # type: ignore

                    if caller_type == "code_execution_20250825":
                        print(f"[PTC] Tool called from code execution environment: {tool_name}")

                    elif caller_type == "direct":
                        print(f"[Direct] Tool called by model: {tool_name}")

                    result = tool_functions[tool_name](**tool_input)

                    # 결과를 API에 적합한 content로 형식 지정
                    if isinstance(result, list) and result and isinstance(result[0], str):
                        content = "\n".join(result)
                    elif isinstance(result, (dict, list)):
                        content = json.dumps(result)
                    else:
                        content = str(result)

                    tool_results.append(
                        {
                            "type": "tool_result",
                            "tool_use_id": tool_use_id,
                            "content": content,
                        }
                    )

            messages.append({"role": "user", "content": tool_results})

        else:
            print(f"\nUnexpected stop reason: {response.stop_reason}")
            elapsed_time = time.time() - start_time

            final_response = next(
                (block.text for block in response.content if isinstance(block, BetaTextBlock)),
                f"Stopped with reason: {response.stop_reason}",
            )
            return final_response, messages, total_tokens, elapsed_time, api_counter

In [9]:
# PTC agent 실행
result_ptc, conversation_ptc, total_tokens_ptc, elapsed_time_ptc, api_count_with_ptc = (
    run_agent_with_ptc(query)
)

╭────────────────────────────────────────────── Claude API Response ──────────────────────────────────────────────╮
│ Claude Message (assistant) │ tokens: 4,134 in • 539 out • 4,673 total                                           │
│ ├── Model: claude-sonnet-4-6                                                                           │
│ ├── Stop Reason: tool_use                                                                                       │
│ └── Content (3 blocks)                                                                                          │
│     ├── Block 1                                                                                                 │
│     │   └── Text                                                                                                │
│     │       └── I'll help you identify which engineering team members exceeded their Q3 travel budget. Let me   │
│     │           start by getting the engineering team members and their expenses.                               │
│     ├── Block 2                                                                                                 │
│     │   └── Server Tool Use                                                                                     │
│     │       ├── ID: srvtoolu_015mWPqaFni4B313UieCxbny                                                           │
│     │       ├── Caller: direct                                                                                  │
│     │       └── Code:                                                                                           │
│     │           └──    1                                                                                        │
│     │                  2 import asyncio                                                                         │
│     │                  3 import json                                                                            │
│     │                  4                                                                                        │
│     │                  5 async def main():                                                                      │
│     │                  6     # First, get all engineering team members                                          │
│     │                  7     team_members_json = await get_team_members({'department': 'engineering'})          │
│     │                  8     team_members = json.loads(team_members_json)                                       │
│     │                  9                                                                                        │
│     │                 10     print(f"Found {len(team_members)} engineering team members")                       │
│     │                 11                                                                                        │
│     │                 12     # Get Q3 expenses for all team members in parallel                                 │
│     │                 13     expense_tasks = [                                                                  │
│     │                 14         get_expenses({'employee_id': member['id'], 'quarter': 'Q3'})                   │
│     │                 15         for member in team_members                                                     │
│     │                 16     ]                                                                                  │
│     │                 17                                                                                        │
│     │                 18     expenses_results = await asyncio.gather(*expense_tasks)                            │
│     │                 19                                                                                        │
│     │                 20     # Calculate travel expenses for each member                                        │
│     │                 21     travel_spending = {}              


[Container] ID: container_011CVSAwq5J4vNPi3A4P2Rwh
[Container] Expires at: 2025-11-24 05:41:17.467494+00:00
[PTC] Tool called from code execution environment: get_team_members


╭──────────────────── Claude API Response ────────────────────╮
│ Claude Message (assistant) │ tokens: 0 in • 0 out • 0 total │
│ ├── Model: claude-sonnet-4-6                       │
│ ├── Stop Reason: tool_use                                   │
│ └── Content (8 blocks)                                      │
│     ├── Block 1                                             │
│     │   └── Tool Use: get_expenses                          │
│     │       ├── ID: toolu_01Nq2au3W69RmDFZaSdqe6u1          │
│     │       ├── Caller: code execution environment          │
│     │       └── Input:                                      │
│     │           └── {                                       │
│     │                 "employee_id": "ENG007",              │
│     │                 "quarter": "Q3"                       │
│     │               }                                       │
│     ├── Block 2                                             │
│     │   └── Tool Use: get_expenses                          │
│     │       ├── ID: toolu_01YYw9cuTSXk7bu7P38qBz6P          │
│     │       ├── Caller: code execution environment          │
│     │       └── Input:                                      │
│     │           └── {                                       │
│     │                 "employee_id": "ENG005",              │
│     │                 "quarter": "Q3"                       │
│     │               }                                       │
│     ├── Block 3                                             │
│     │   └── Tool Use: get_expenses                          │
│     │       ├── ID: toolu_01Fyxxe2KmVpVmw4jJL2CXSz          │
│     │       ├── Caller: code execution environment          │
│     │       └── Input:                                      │
│     │           └── {                                       │
│     │                 "employee_id": "ENG008",              │
│     │                 "quarter": "Q3"                       │
│     │               }                                       │
│     ├── Block 4                                             │
│     │   └── Tool Use: get_expenses                          │
│     │       ├── ID: toolu_01J4ovDu2UJa9Se19vxKTa6y          │
│     │       ├── Caller: code execution environment          │
│     │       └── Input:                                      │
│     │           └── {                                       │
│     │                 "employee_id": "ENG006",              │
│     │                 "quarter": "Q3"                       │
│     │               }                                       │
│     ├── Block 5                                             │
│     │   └── Tool Use: get_expenses                          │
│     │       ├── ID: toolu_01T24CrvQYA3LqGfZftCmueC          │
│     │       ├── Caller: code execution environment          │
│     │       └── Input:                                      │
│     │           └── {                                       │
│     │                 "employee_id": "ENG003",              │
│     │                 "quarter": "Q3"                       │
│     │               }                                       │
│     ├── Block 6                                             │
│     │   └── Tool Use: get_expenses                          │
│     │       ├── ID: toolu_01HotYZN6sk3gMLkpdWXbdz4          │
│     │       ├── Caller: code execution environment          │
│     │       └── Input:                                      │
│     │           └── {                                       │
│     │                 "employee_id": "ENG004",              │
│     │                 "quarter": "Q3"                       │
│     │               }                                       │
│     ├── Block 7                                             │
│     │   └── Tool Use: get_expenses                          │
│     │       ├── ID: toolu_01AxvEqi3AKqdnH44kGH1U6E          │
│     │       ├── Caller: code execution 


[Container] ID: container_011CVSAwq5J4vNPi3A4P2Rwh
[Container] Expires at: 2025-11-24 05:41:19.266670+00:00
[PTC] Tool called from code execution environment: get_expenses
[PTC] Tool called from code execution environment: get_expenses
[PTC] Tool called from code execution environment: get_expenses
[PTC] Tool called from code execution environment: get_expenses
[PTC] Tool called from code execution environment: get_expenses
[PTC] Tool called from code execution environment: get_expenses
[PTC] Tool called from code execution environment: get_expenses
[PTC] Tool called from code execution environment: get_expenses


╭─────────────────────────────────────── Claude API Response ───────────────────────────────────────╮
│ Claude Message (assistant) │ tokens: 4,751 in • 679 out • 5,430 total                             │
│ ├── Model: claude-sonnet-4-6                                                             │
│ ├── Stop Reason: tool_use                                                                         │
│ └── Content (6 blocks)                                                                            │
│     ├── Block 1                                                                                   │
│     │   └── Code Execution Result: Success (exit 0)                                               │
│     │       └── stdout:                                                                           │
│     │           └── Found 8 engineering team members                                              │
│     │                                                                                             │
│     │               Employees who exceeded $5,000 standard budget: 3                              │
│     │               ENG001: Alice Chen - $9177.88                                                 │
│     │               ENG003: Carol White - $6483.49                                                │
│     │               ENG007: Grace Taylor - $5289.35                                               │
│     │                                                                                             │
│     ├── Block 2                                                                                   │
│     │   └── Text                                                                                  │
│     │       └── Now let me check if any of these three employees have custom budget exceptions:   │
│     ├── Block 3                                                                                   │
│     │   └── Server Tool Use                                                                       │
│     │       ├── ID: srvtoolu_015GTpFmCbd2JPAQLAioB4Qb                                             │
│     │       ├── Caller: direct                                                                    │
│     │       └── Code:                                                                             │
│     │           └──    1                                                                          │
│     │                  2 import asyncio                                                           │
│     │                  3 import json                                                              │
│     │                  4                                                                          │
│     │                  5 async def main():                                                        │
│     │                  6     # Check custom budgets for the three employees who exceeded standard │
│     │                  7     exceeded_ids = ['ENG001', 'ENG003', 'ENG007']                        │
│     │                  8     exceeded_amounts = {                                                 │
│     │                  9         'ENG001': {'name': 'Alice Chen', 'travel_total': 9177.88},       │
│     │                 10         'ENG003': {'name': 'Carol White', 'travel_total': 6483.49},      │
│     │                 11         'ENG007': {'name': 'Grace Taylor', 'travel_total': 5289.35}      │
│     │                 12     }                                                                    │
│     │                 13                                                                          │
│     │                 14     # Get custom budgets in parallel                                     │
│     │                 15     budget_tasks = [                                                     │
│     │                 16         get_custom_budget({'user_id': emp_id})                           │
│     │                 17     


[Container] ID: container_011CVSAwq5J4vNPi3A4P2Rwh
[Container] Expires at: 2025-11-24 05:41:33.430636+00:00
[PTC] Tool called from code execution environment: get_custom_budget
[PTC] Tool called from code execution environment: get_custom_budget
[PTC] Tool called from code execution environment: get_custom_budget


╭────────────────────────────────────────────── Claude API Response ──────────────────────────────────────────────╮
│ Claude Message (assistant) │ tokens: 5,611 in • 205 out • 5,816 total                                           │
│ ├── Model: claude-sonnet-4-6                                                                           │
│ ├── Stop Reason: end_turn                                                                                       │
│ └── Content (2 blocks)                                                                                          │
│     ├── Block 1                                                                                                 │
│     │   └── Code Execution Result: Success (exit 0)                                                             │
│     │       └── stdout:                                                                                         │
│     │           └── ENGINEERING TEAM MEMBERS WHO EXCEEDED THEIR Q3 TRAVEL BUDGET:                               │
│     │               ================================================================================            │
│     │                                                                                                           │
│     │               Alice Chen (ENG001)                                                                         │
│     │                 Budget Limit: $5,000.00 (Standard)                                                        │
│     │                 Travel Spending: $9,177.88                                                                │
│     │                 Exceeded By: $4,177.88                                                                    │
│     │                                                                                                           │
│     │               Carol White (ENG003)                                                                        │
│     │                 Budget Limit: $5,000.00 (Standard)                                                        │
│     │                 Travel Spending: $6,483.49                                                                │
│     │                 Exceeded By: $1,483.49                                                                    │
│     │                                                                                                           │
│     │               Grace Taylor (ENG007)                                                                       │
│     │                 Budget Limit: $5,000.00 (Standard)                                                        │
│     │                 Travel Spending: $5,289.35                                                                │
│     │                 Exceeded By: $289.35                                                                      │
│     │                                                                                                           │
│     └── Block 2                                                                                                 │
│         └── Text                                                                                                │
│             └── ## Summary                                                                                      │
│                                                                                                                 │
│                 **Three engineering team members exceeded their Q3 travel budget:**                             │
│                                                                                                                 │
│                 1. **Alice Chen (ENG001)**                                                                      │
│                    - Budget: $5,000 (Standard)                                                                  │
│                    - Spent: $9,177.88                          

In [10]:
print(f"\n{'=' * 60}")
print(f"Result: {result_ptc}")
print(f"\n{'=' * 60}")
print("Performance Metrics:")
print(
    f"  Total API calls to Claude: {len([m for m in conversation_ptc if m['role'] == 'assistant'])}"
)
print(f"  Total tokens used: {total_tokens_ptc:,}")
print(f"  Total time taken: {elapsed_time_ptc:.2f}s")


Result: ## Summary

**Three engineering team members exceeded their Q3 travel budget:**

1. **Alice Chen (ENG001)**
   - Budget: $5,000 (Standard)
   - Spent: $9,177.88
   - Over budget by: **$4,177.88**

2. **Carol White (ENG003)**
   - Budget: $5,000 (Standard)
   - Spent: $6,483.49
   - Over budget by: **$1,483.49**

3. **Grace Taylor (ENG007)**
   - Budget: $5,000 (Standard)
   - Spent: $5,289.35
   - Over budget by: **$289.35**

All three employees are on the standard $5,000 quarterly travel budget with no custom exceptions, so they all genuinely exceeded their allocated travel budget for Q3.

Performance Metrics:
  Total API calls to Claude: 3
  Total tokens used: 15,919
  Total time taken: 34.88s


## 성능 비교

기존 tool calling과 PTC의 성능을 비교하겠습니다.

**API 호출 수에 관한 참고:** 이 예제에서는 PTC에 더 많은 API 호출이 필요하다는 것을 알 수 있습니다. 이는 PTC가 모범 사례에 따라 더 구조화된 순차적 코드를 작성하기 때문입니다. 예를 들어 경비 조회 단계와 예산 확인 단계를 분리합니다. 기존 tool calling은 때때로 단일 turn에서 작업을 batch로 묶을 수 있지만, 모든 원시 데이터를 모델의 context로 보내야 하는 비용이 있습니다. 특히 metadata가 풍부한 큰 dataset을 다룰 때 PTC의 token 효율성 이점은 round-trip의 작은 증가를 훨씬 능가합니다.


In [11]:
import pandas as pd

# 비교 dataframe 생성
comparison_data = {
    "Metric": [
        "API Calls",
        "Total Tokens",
        "Elapsed Time (s)",
        "Token Reduction",
        "Time Reduction",
    ],
    "Traditional": [
        api_count_without_ptc,
        f"{total_tokens:,}",
        f"{elapsed_time:.2f}",
        "-",
        "-",
    ],
    "PTC": [
        api_count_with_ptc,
        f"{total_tokens_ptc:,}",
        f"{elapsed_time_ptc:.2f}",
        f"{((total_tokens - total_tokens_ptc) / total_tokens * 100):.1f}%",
        f"{((elapsed_time - elapsed_time_ptc) / elapsed_time * 100):.1f}%",
    ],
}

df = pd.DataFrame(comparison_data)
print(df.to_string(index=False))

          Metric Traditional    PTC
       API Calls           4      4
    Total Tokens     110,473 15,919
Elapsed Time (s)       35.38  34.88
 Token Reduction           -  85.6%
  Time Reduction           -   1.4%


## 핵심 내용

이 예제에서 PTC는 세 가지 핵심 기능을 통해 상당한 성능 개선을 보여주었습니다.

### 1. 큰 데이터 Parsing을 통한 Context 보존
이것이 workflow에서 시연한 주된 이점입니다. Claude는 code execution 환경 안에서 수백 개의 경비 항목을 가져와 처리하는 코드를 작성했습니다. 데이터를 programmatically 처리하여 Claude는 JSON parsing, 상태별 filtering, 범주별 금액 합산, 예산 한도 비교를 수행했습니다. 원시 경비 데이터와 metadata를 모델의 context window로 보내지 않고 이 모든 것을 수행하여 **token 사용량을 크게 줄였습니다**.

### 2. 순차적 Dependency 최적화
API에는 순차적인 dependency가 있습니다. `get_custom_budget(user_id)`는 경비를 분석하여 표준 $5,000 예산을 초과한 사람을 파악한 후에만 호출할 수 있습니다. 기존 tool calling에서는 팀원 조회, 각 사람의 경비 조회, 예산 초과자 파악, 사용자 정의 예산을 하나씩 확인하는 여러 round-trip이 필요합니다. PTC에서는 Claude가 code execution 환경에서 이 전체 workflow를 orchestration하는 코드를 작성하여, loop 안에서 programmatic tool call을 수행하고 호출 간 state를 유지합니다. 이로써 많았을 순차적 API round-trip을 더 똑똑한 orchestration을 통한 더 적은 호출로 바꿉니다.

### 3. Code Execution에서의 계산 Logic
복잡한 metadata가 있는 수십 개의 경비를 모델이 머릿속으로 추적하고 합산하도록 하는 대신, Claude는 산술과 집계 logic을 Python 코드에 맡겼습니다. 이는 모델의 인지적 부담을 줄이고, 정확한 계산을 보장하며, 관련 없는 metadata(영수증 URL과 가맹점 위치 등)를 모델의 context에서 완전히 제외했습니다.

---

## PTC를 사용할 때

PTC는 다음 경우에 가장 유용합니다.

- Filtering, parsing, 집계가 필요한 **metadata가 풍부한 큰 dataset 처리**(영수증 URL, 승인 chain, 가맹점 세부사항 등이 있는 경비 분석 예제처럼)
- 한 tool 호출이 이전 호출 결과에 의존하는 **순차적 dependency 존재**(표준 한도를 초과한 직원만 사용자 정의 예산을 확인하는 경우처럼)
- 비슷한 entity에 대해 순차적으로 또는 loop 안에서 **여러 tool 호출 필요**(각 팀원의 경비와 예산 확인)
- **계산 logic**으로 모델 context를 거쳐야 하는 내용을 줄일 수 있음
- 사람이 감독하지 않아도 programmatic/반복 실행에 **tool이 안전함**

## 결론

팀 경비 분석은 PTC의 강점을 보여주었습니다. **Metadata가 풍부한 큰 dataset을 다룰 때 context 소모를 획기적으로 줄이고**, **순차적 dependency가 있는 workflow를 최적화합니다**. Claude가 tool 호출을 orchestration하고 결과를 programmatically 처리하는 코드를 작성하게 하여, 정확도와 분석 품질을 유지하면서 상당한 token 절감을 달성했습니다.

PTC는 풍부한 metadata를 포함하는 대량 데이터 처리, dependency가 있는 반복 tool 호출, 또는 원시 tool output이 모델 context를 오염시킬 수 있는 시나리오의 workflow에 특히 유용합니다.

## 다음 단계

자신의 사용 사례에 이 패턴을 적용해 보세요.
- 순차 조회를 통한 금융 데이터 분석과 보고
- 초기 scan 결과에 의존하는 여러 entity의 상태 확인
- Metadata가 있는 큰 파일 처리(CSV, JSON, XML parsing)
- 후속 query를 통한 database query 결과 집계
- 초기 결과에 따른 조건 logic이 있는 batch API 작업
